# Structured Streaming — Wikimedia recent changes

TP 05 — ECE Big Data Processing. Ce notebook reprend la démo (agrégation des éditions
par bot) puis l'enrichit :

| # | Exploration | Fenêtre | Mode de sortie |
|---|---|---|---|
| 0 | Démo : éditions par bot | tumbling 1 h | complete |
| 1 | Activité et taille des éditions par minute | **tumbling** 1 min | update + watermark |
| 2 | Volume par wiki / type d'auteur, glissant | **sliding** 5 min, pas 1 min | update + watermark |
| 3 | Alerte « grosses modifications » (\|Δ\| ≥ 1000 o) | aucune (stateless) | append |
| 4 | Top pages / top contributeurs | **sliding** 10 min, pas 2 min | complete → memory |

Les requêtes `console` s'affichent dans les logs Docker : `docker logs -f pyspark_notebook`.

In [1]:
# Imports cell
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, window, count, sum as _sum, avg, max as _max, min as _min,
    abs as _abs, when, approx_count_distinct, round as _round, substring,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType, BooleanType,
)

In [2]:
# initialize the spark session
# Le connecteur Kafka doit avoir EXACTEMENT la même version que Spark (sinon NoSuchMethodError).
# On la lit donc depuis pyspark au lieu de l'écrire en dur (l'image actuelle embarque Spark 4.2.0).
import pyspark
KAFKA_PACKAGE = f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}"
print("Spark", pyspark.__version__, "->", KAFKA_PACKAGE)

spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", KAFKA_PACKAGE) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Moins de bruit dans les logs pour bien voir les sorties console
spark.sparkContext.setLogLevel("WARN")

Spark 4.2.0 -> org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0


In [3]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()

## Schéma et préparation

On garde le schéma de la démo et on ajoute `namespace`, `server_name` et `meta.domain`.
Colonnes dérivées :
- `event_time` : `timestamp` (epoch en secondes) converti en vrai type `timestamp`
  (`from_unixtime` de la démo renvoie une *chaîne*, que `window` doit recaster) ;
- `size_delta` = `length.new - length.old` (octets ajoutés ou retirés) et `abs_delta`.

In [4]:
wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True),
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), wikimedia_schema).alias("data")
).select("data.*")

# Colonnes dérivées utilisées par toutes les explorations
events_df = parsed_df \
    .withColumn("event_time", col("timestamp").cast("timestamp")) \
    .withColumn("size_delta",
                col("length.new").cast("long") - when(col("length.old").isNull(), 0)
                                                   .otherwise(col("length.old"))) \
    .withColumn("abs_delta", _abs(col("size_delta"))) \
    .withColumn("author_type", when(col("bot"), "bot").otherwise("human"))

events_df.printSchema()

root
 |-- title: string (nullable = true)
 |-- user: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- server_name: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- comment: string (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: integer (nullable = true)
 |    |-- new: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- size_delta: long (nullable = true)
 |-- abs_delta: long (nullable = true)
 |-- author_type: string (nullable = false)



## 0. Démo : nombre d'éditions par bot (tumbling 1 h, complete)

In [5]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = events_df \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

# Use console output (goes to Docker logs)
demo_query = windowed_counts_df \
    .writeStream \
    .queryName("demo_bot_counts") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## 1. Fenêtres **tumbling** (1 min) : activité et taille des éditions

Fenêtres fixes **sans recouvrement** : chaque événement tombe dans exactement une fenêtre.
Pour chaque minute : nombre d'éditions, part des bots, contributeurs distincts,
octets ajoutés nets, taille moyenne et maximale d'une modification.

Le **watermark** de 2 min dit à Spark qu'un événement arrivant avec plus de 2 min de retard
peut être ignoré : l'état des fenêtres anciennes est alors purgé (sinon il grossit sans fin).
Mode `update` : à chaque micro-batch on n'affiche que les fenêtres modifiées.

In [6]:
tumbling_df = events_df \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(window(col("event_time"), "1 minute")) \
    .agg(
        count("*").alias("edits"),
        _sum(when(col("bot"), 1).otherwise(0)).alias("bot_edits"),
        approx_count_distinct("user").alias("distinct_users"),
        _sum("size_delta").alias("net_bytes"),
        _round(avg("abs_delta"), 1).alias("avg_abs_delta"),
        _max("size_delta").alias("biggest_add"),
        _min("size_delta").alias("biggest_removal"),
    ) \
    .withColumn("bot_share_pct", _round(100 * col("bot_edits") / col("edits"), 1)) \
    .select(col("window.start").alias("window_start"), col("window.end").alias("window_end"),
            "edits", "bot_edits", "bot_share_pct", "distinct_users",
            "net_bytes", "avg_abs_delta", "biggest_add", "biggest_removal")

tumbling_query = tumbling_df \
    .writeStream \
    .queryName("tumbling_1min_activity") \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## 2. Fenêtres **sliding** (5 min, pas de 1 min) : volume par wiki et type d'auteur

Fenêtres de 5 min qui démarrent toutes les minutes → elles **se recouvrent** :
chaque événement est compté dans 5 fenêtres. On obtient une moyenne mobile qui lisse les pics.
Avec plusieurs wikis dans le producteur (`--wiki fr.wikipedia.org --wiki de.wikipedia.org`),
on compare leur activité.

In [7]:
sliding_df = events_df \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(
        window(col("event_time"), "5 minutes", "1 minute"),
        col("server_name"),
        col("author_type"),
    ) \
    .agg(
        count("*").alias("edits"),
        _sum("abs_delta").alias("bytes_changed"),
        _sum(when(col("minor"), 1).otherwise(0)).alias("minor_edits"),
    ) \
    .withColumn("edits_per_min", _round(col("edits") / 5, 2)) \
    .select(col("window.start").alias("window_start"), col("window.end").alias("window_end"),
            "server_name", "author_type", "edits", "edits_per_min", "minor_edits", "bytes_changed")

sliding_query = sliding_df \
    .writeStream \
    .queryName("sliding_5min_by_wiki") \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .trigger(processingTime="30 seconds") \
    .start()

## 3. Alerte : grosses modifications (stateless, append)

Pas d'agrégation ici : un simple `filter`. Spark n'a aucun état à garder, chaque événement
est traité une seule fois → mode `append`. Utile pour repérer vandalismes (grosses suppressions)
ou gros ajouts de contenu.

In [8]:
BIG_EDIT_BYTES = 1000

big_edits_df = events_df \
    .filter(col("abs_delta") >= BIG_EDIT_BYTES) \
    .withColumn("kind", when(col("size_delta") > 0, "ADD").otherwise("REMOVAL")) \
    .withColumn("comment", substring(col("comment"), 1, 60)) \
    .select("event_time", "server_name", "title", "user", "author_type", "kind", "size_delta", "comment")

big_edits_query = big_edits_df \
    .writeStream \
    .queryName("big_edits_alert") \
    .format("console") \
    .outputMode("append") \
    .option("truncate", False) \
    .start()

## 4. Top pages et top contributeurs (sliding 10 min, pas 2 min) → table en mémoire

Le sink `memory` stocke le résultat dans une table interrogeable en SQL depuis le notebook :
pratique pour explorer sans passer par les logs. Le tri n'est autorisé en streaming qu'en
mode `complete` après agrégation, on trie donc au moment de la requête SQL.

In [9]:
top_pages_query = events_df \
    .groupBy(window(col("event_time"), "10 minutes", "2 minutes"), col("title")) \
    .agg(count("*").alias("edits"), approx_count_distinct("user").alias("editors"),
         _sum("size_delta").alias("net_bytes")) \
    .writeStream \
    .queryName("top_pages") \
    .format("memory") \
    .outputMode("complete") \
    .start()

top_users_query = events_df \
    .groupBy(window(col("event_time"), "10 minutes", "2 minutes"), col("user"), col("author_type")) \
    .agg(count("*").alias("edits"), _sum("abs_delta").alias("bytes_changed")) \
    .writeStream \
    .queryName("top_users") \
    .format("memory") \
    .outputMode("complete") \
    .start()

In [12]:
# À relancer quand on veut : affiche le top 10 de la fenêtre la plus récente
latest_pages = spark.sql("""
    SELECT window.start AS window_start, window.end AS window_end, title, edits, editors, net_bytes
    FROM top_pages
    WHERE window.end = (SELECT max(window.end) FROM top_pages)
    ORDER BY edits DESC, editors DESC
    LIMIT 10
""")
latest_pages.show(truncate=False)

latest_users = spark.sql("""
    SELECT window.start AS window_start, window.end AS window_end, user, author_type, edits, bytes_changed
    FROM top_users
    WHERE window.end = (SELECT max(window.end) FROM top_users)
    ORDER BY edits DESC
    LIMIT 10
""")
latest_users.show(truncate=False)

+-------------------+-------------------+-----------------------------------------------+-----+-------+---------+
|window_start       |window_end         |title                                          |edits|editors|net_bytes|
+-------------------+-------------------+-----------------------------------------------+-----+-------+---------+
|2026-10-08 14:44:00|2026-10-08 14:54:00|Ligue des champions féminine de l'EHF 2026-2027|2    |1      |14       |
|2026-10-08 14:44:00|2026-10-08 14:54:00|Joaquín Freitas                                |2    |1      |700      |
|2026-10-08 14:44:00|2026-10-08 14:54:00|Bergueneuse                                    |2    |1      |1125     |
|2026-10-08 14:44:00|2026-10-08 14:54:00|Utilisateur:Poro amara/Brouillon               |1    |1      |105      |
|2026-10-08 14:44:00|2026-10-08 14:54:00|Ail rose du Var                                |1    |1      |91       |
|2026-10-08 14:44:00|2026-10-08 14:54:00|Nelson (Colombie-Britannique)                  

## Suivi et arrêt des requêtes

In [11]:
# État et dernière progression (nb de lignes lues, débit, watermark courant) de chaque requête
for q in spark.streams.active:
    p = q.lastProgress or {}
    print(f"{q.name:<25} active={q.isActive}  rows/s={p.get('processedRowsPerSecond')}  "
          f"watermark={p.get('eventTime', {}).get('watermark')}")

demo_bot_counts           active=True  rows/s=None  watermark=None
big_edits_alert           active=True  rows/s=None  watermark=None
tumbling_1min_activity    active=True  rows/s=None  watermark=None
sliding_5min_by_wiki      active=True  rows/s=None  watermark=None
top_users                 active=True  rows/s=None  watermark=None
top_pages                 active=True  rows/s=None  watermark=None
